# Gap Fill Mean Reversion on SPY
## Strategy Brief
The Gap Fill Mean Reversion strategy aims to capitalize on the tendency of price gaps to "fill" after the market opens. A price gap occurs when the opening price is significantly higher or lower than the previous closing price. The strategy predicts that the price will revert to the previous day's close, filling the gap. Trades are executed when a gap is identified, with positions taken to profit from the expected price movement back to the prior close. Historical testing on SPY shows that this strategy can be profitable, but it requires careful risk management due to potential for large gaps not filling.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants that will be used throughout the strategy. These include the ticker symbol for SPY, the start date for data collection, and any thresholds or limits for the strategy.

In [ ]:
# Configuration
TICKER = 'SPY'
START_DATE = '2010-01-01'
END_DATE = None  # Use None to indicate up to the current date
GAP_THRESHOLD = 0.005  # 0.5% gap threshold for triggering trades

## PHASE 2 - Data Exploration
In this phase, we download historical price data for SPY using yfinance. We then calculate the gap size and plot it alongside the price to visualize when gaps occur.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
spy_data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate gap size
spy_data['Gap'] = (spy_data['Open'] - spy_data['Close'].shift(1)) / spy_data['Close'].shift(1)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(spy_data['Close'], label='SPY Close')
plt.plot(spy_data['Gap'], label='Gap', linestyle='--')
plt.title('SPY Price and Gap Size')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We define the strategy's signal based on the gap size. If the gap exceeds the threshold, a trade is triggered. We then define the entry and exit logic, and create a positions series indicating when we are in a trade.

In [ ]:
# Define signal
spy_data['Signal'] = np.where(spy_data['Gap'].abs() > GAP_THRESHOLD, -np.sign(spy_data['Gap']), 0)

# Define positions
spy_data['Position'] = spy_data['Signal'].shift(1)

# Fill initial NaN position with 0
spy_data['Position'].fillna(0, inplace=True)

## PHASE 4 - Coding & Backtesting
We backtest the strategy by calculating daily returns based on the positions. We then plot the equity curve to visualize the strategy's performance over time.

In [ ]:
# Calculate daily returns
spy_data['Market_Return'] = spy_data['Close'].pct_change()
spy_data['Strategy_Return'] = spy_data['Position'] * spy_data['Market_Return']

# Calculate equity curve
spy_data['Equity_Curve'] = (1 + spy_data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(spy_data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We also compare these metrics to a simple buy-and-hold strategy.

In [ ]:
# Performance metrics
cagr = (spy_data['Equity_Curve'].iloc[-1]) ** (1 / ((spy_data.index[-1] - spy_data.index[0]).days / 365.25)) - 1
sharpe_ratio = spy_data['Strategy_Return'].mean() / spy_data['Strategy_Return'].std() * np.sqrt(252)
negative_returns = spy_data['Strategy_Return'][spy_data['Strategy_Return'] < 0]
sortino_ratio = spy_data['Strategy_Return'].mean() / negative_returns.std() * np.sqrt(252)
max_drawdown = (spy_data['Equity_Curve'].cummax() - spy_data['Equity_Curve']).max()
calmar_ratio = cagr / max_drawdown

# Buy-and-hold metrics
bh_cagr = (spy_data['Close'].iloc[-1] / spy_data['Close'].iloc[0]) ** (1 / ((spy_data.index[-1] - spy_data.index[0]).days / 365.25)) - 1
bh_max_drawdown = (spy_data['Close'].cummax() - spy_data['Close']).max()

# Comparison table
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy & Hold': [bh_cagr, np.nan, np.nan, np.nan, bh_max_drawdown]
})
print(performance_df)

## PHASE 6 - Deploy & Monitor
We define a function to download the last 60 days of data and compute today's signal. This allows us to monitor the strategy and determine whether to enter a position today.

In [ ]:
def check_today_signal():
    # Download last 60 days of data
    recent_data = yf.download(TICKER, period='60d')
    
    # Calculate gap for today
    recent_data['Gap'] = (recent_data['Open'] - recent_data['Close'].shift(1)) / recent_data['Close'].shift(1)
    
    # Today's signal
    today_signal = np.where(recent_data['Gap'].iloc[-1].abs() > GAP_THRESHOLD, -np.sign(recent_data['Gap'].iloc[-1]), 0)
    
    print(f"Today's Signal: {today_signal}")

# Run the function to check today's signal
check_today_signal()